<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-09-multimodal-and-pretrained/lesson-9.1-multimodal/notebooks/GCP_Capstone_9.1_Multimodal.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 9.1 Gemini Multimodal — Images, Audio, Video & PDFs
**Netsetos GenAI Engineering — GCP Capstone**


In [ ]:
!pip install -q google-genai Pillow reportlab
from google import genai
from google.genai import types
import pathlib, time, json

# WHY THE DEVELOPER API HERE (and not Vertex):
# This lesson notebook demonstrates the File API (client.files.upload / files.get /
# files.delete in Cell 5), which is a Gemini *Developer API* feature. Vertex AI does
# NOT expose files.upload() — on Vertex you stage large media in Cloud Storage and
# pass types.Part.from_uri('gs://...') instead. The Practice Lab uses that production
# path: genai.Client(enterprise=True, project=..., location=...) + GCS URIs (ADC via
# auth.authenticate_user(), no API keys). Use Vertex in real DocuMind; the swap is:
#   from google.colab import auth; auth.authenticate_user()
#   client = genai.Client(enterprise=True, project='documind-ai-YOUR-ID', location='us-central1')
client = genai.Client(api_key='YOUR_KEY')  # Developer API — enables the File API demo below
print('SDK ready:', genai.__version__)

## Cell 1: Image Analysis — Invoice OCR


In [ ]:
# Download a sample invoice image
import urllib.request
url = 'https://upload.wikimedia.org/wikipedia/commons/0/0b/ReceiptSwiss.jpg'
urllib.request.urlretrieve(url, 'receipt.jpg')

img_bytes = pathlib.Path('receipt.jpg').read_bytes()
print(f'Image size: {len(img_bytes)//1024} KB')

response = client.models.generate_content(
    model='gemini-3.6-flash',
    contents=[
        types.Part.from_bytes(data=img_bytes, mime_type='image/jpeg'),
        'Extract all text from this receipt. Return vendor, date, items, total as JSON.'
    ],
    config=types.GenerateContentConfig(
        response_mime_type='application/json',
        temperature=0.1
    )
)
print(response.text)


## Cell 2: Token Counting


In [ ]:
# Count tokens for the image
token_count = client.models.count_tokens(
    model='gemini-3.6-flash',
    contents=[
        types.Part.from_bytes(data=img_bytes, mime_type='image/jpeg'),
        'Extract text'
    ]
)
print(f'Image + prompt tokens: {token_count.total_tokens}')
print(f'Estimated cost (Flash): ${token_count.total_tokens * 1.50 / 1_000_000:.6f}')

# Estimate tokens for different modalities
def estimate_tokens(modality, value):
    if modality == 'image_small': return 258
    if modality == 'image_1024': return 1290
    if modality == 'pdf_pages': return value * 258
    if modality == 'audio_minutes': return value * 1920
    if modality == 'video_minutes': return value * 18000

for name, val in [('image_small', 1), ('image_1024', 1), ('pdf_pages', 50), ('audio_minutes', 30), ('video_minutes', 5)]:
    tokens = estimate_tokens(name, val)
    cost = tokens * 1.50 / 1_000_000
    print(f'{name}({val}): {tokens:,} tokens = ${cost:.4f}')


## Cell 3: PDF Processing


In [ ]:
# Create a sample PDF
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas
try:
    c = canvas.Canvas('sample.pdf', pagesize=letter)
    c.drawString(72, 700, 'Invoice #2024-001')
    c.drawString(72, 680, 'Vendor: TechCorp Solutions')
    c.drawString(72, 660, 'Date: 2024-03-15')
    c.drawString(72, 640, 'Item: Cloud Services - $4,500')
    c.drawString(72, 620, 'Item: Support Package - $1,200')
    c.drawString(72, 600, 'Total: $5,700')
    c.save()
    print('PDF created')
except ImportError:
    print('Install reportlab: pip install reportlab')
    # Fallback: use the image

# Analyze PDF
pdf_bytes = pathlib.Path('sample.pdf').read_bytes()
response = client.models.generate_content(
    model='gemini-3.6-flash',
    contents=[
        types.Part.from_bytes(data=pdf_bytes, mime_type='application/pdf'),
        'Extract all invoice details as JSON.'
    ],
    config=types.GenerateContentConfig(
        response_mime_type='application/json',
        temperature=0.1
    )
)
print(response.text)


## Cell 4: PIL Image Analysis


In [ ]:
from PIL import Image
import io

# Create a simple test image
img = Image.new('RGB', (384, 384), color='white')
from PIL import ImageDraw
draw = ImageDraw.Draw(img)
draw.text((50, 150), 'APPROVED', fill='green')
draw.text((50, 200), 'Amount: Rs. 45,000', fill='black')
img.save('stamp.png')

# Send PIL image directly
response = client.models.generate_content(
    model='gemini-3.6-flash',
    contents=[img, 'What does this stamp say? Extract the amount.']
)
print(response.text)


## Cell 5: File API for Large Files


In [ ]:
# Demonstrate File API pattern (would use real audio/video in production)
# This shows the upload + poll + generate pattern

# Upload
print('Uploading file...')
uploaded = client.files.upload(file='sample.pdf')
print(f'Uploaded: {uploaded.name}')
print(f'State: {uploaded.state}')

# Poll until active (instant for small files, seconds for video)
while hasattr(uploaded.state, 'name') and uploaded.state.name != 'ACTIVE':
    time.sleep(2)
    uploaded = client.files.get(name=uploaded.name)
    print(f'State: {uploaded.state}')

print('File is ACTIVE')

# Generate
response = client.models.generate_content(
    model='gemini-3.6-flash',
    contents=[uploaded, 'Summarize this document.']
)
print(response.text)

# Cleanup
client.files.delete(name=uploaded.name)
print('File deleted')


## Cell 6: Error Handling Pattern


In [ ]:
import random
from google.genai import errors

def analyze_with_retry(client, contents, max_retries=3):
    for attempt in range(max_retries):
        try:
            return client.models.generate_content(
                model='gemini-3.6-flash',
                contents=contents
            )
        except errors.APIError as e:
            if e.code == 429:  # rate limited
                wait = (2 ** attempt) + random.uniform(0, 1)
                print(f'Rate limited. Waiting {wait:.1f}s...')
                time.sleep(wait)
            elif e.code == 503:  # service unavailable
                print('Service unavailable. Waiting 10s...')
                time.sleep(10)
            elif e.code == 400:  # invalid input, no retry
                print(f'Invalid input (no retry): {e.message}')
                raise
            else:
                raise
    raise RuntimeError('Max retries exceeded')

# Test it
result = analyze_with_retry(client, [
    types.Part.from_bytes(data=img_bytes, mime_type='image/jpeg'),
    'Describe this image briefly.'
])
print(result.text)


## Cell 7: Cost Estimation Dashboard


In [ ]:
# DocuMind monthly cost estimate
scenarios = [
    ('500 invoices (1024x1024)', 500 * 1290, 'flash'),
    ('50 contracts (avg 30 pages)', 50 * 30 * 258, 'flash'),
    ('20 meetings (avg 45 min)', 20 * 45 * 1920, 'flash'),
    ('5 video reviews (avg 10 min)', 5 * 10 * 18000, 'flash'),
    ('100 triage classifications', 100 * 258, 'flash-lite'),
]

prices = {'flash': 1.50, 'flash-lite': 0.25, 'pro': 2.00}
total_cost = 0
print('DocuMind Monthly Multimodal Cost Estimate:')
print('=' * 60)
for name, tokens, model in scenarios:
    cost = tokens * prices[model] / 1_000_000
    total_cost += cost
    print(f'{name}')
    print(f'  Tokens: {tokens:>12,}  Model: {model:<10}  Cost: ${cost:.4f}')
print('=' * 60)
print(f'TOTAL: ${total_cost:.4f}/month')
print(f'With context caching (90% savings): ${total_cost * 0.1:.4f}/month')


## Done!
- google-genai SDK (new patterns)
- Image analysis with JSON extraction
- PDF native vision processing
- Token counting and cost estimation
- File API upload/poll/generate pattern
- Production error handling with retries
